In [ ]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score, precision_score, recall_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from transformers import EvalPrediction
from datasets import Dataset, DatasetDict

In [ ]:
# ---- 1. Load and Prepare the Data

df = pd.read_excel('ground_truth_filtered.xlsx')

# Define the text column and extract the label columns
text_col = 'Headline_GER'
label_cols = [col for col in df.columns if col != text_col]

# Convert the 14 label columns into a single list of floats for each row
# (Float is required for BCEWithLogitsLoss in multi-label classification)
df['labels'] = df[label_cols].values.astype(np.float32).tolist()

# Keep only the necessary columns for training
df_clean = df[[text_col, 'labels']].copy()

In [ ]:
# ---- 2. Apply the Train / Dev / Test Splits

# Initial 80/20 Train-Rest split
train_df, rest_df = train_test_split(df_clean, test_size=0.20, random_state=42)

# Splitting the Rest (20%) into 60% Dev and 40% Test
dev_df, test_df = train_test_split(rest_df, test_size=0.40, random_state=42)

print(f"Train size: {len(train_df)}")
print(f"Dev size:   {len(dev_df)}")
print(f"Test size:  {len(test_df)}")

# Convert Pandas DataFrames into Hugging Face Datasets
dataset = DatasetDict({
    'train': Dataset.from_pandas(train_df, preserve_index=False),
    'dev': Dataset.from_pandas(dev_df, preserve_index=False),
    'test': Dataset.from_pandas(test_df, preserve_index=False)
})

In [ ]:
# ---- 3. Tokenization

model_id = "LSX-UniWue/ModernGBERT_134M"
# model_id = "distilbert/distilbert-base-german-cased"
tokenizer = AutoTokenizer.from_pretrained(model_id)

def tokenize_function(examples):
    # Truncate and pad to a standard max length for headlines
    return tokenizer(examples[text_col], padding="max_length", truncation=True, max_length=128)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

In [ ]:
# ---- 4. Model Initialization

num_labels = len(label_cols)

# Load the model configuration first
from transformers import AutoConfig
config = AutoConfig.from_pretrained(model_id)

# Set the dropout probabilities in the config
config.hidden_dropout_prob = 0.10          # Encoder hidden dropout
config.attention_probs_dropout_prob = 0.10 # Attention dropout
config.classifier_dropout_prob = 0.20      # Classification-head dropout

# Set the number of labels and problem type in the config
config.num_labels = num_labels
config.problem_type = "multi_label_classification"

# Load the model with the modified configuration
model_regular = AutoModelForSequenceClassification.from_pretrained(
    model_id,
    config=config
)

In [ ]:
# ---- 5. Define Evaluation Metrics

def compute_metrics(p: EvalPrediction):
    predictions = p.predictions[0] if isinstance(p.predictions, tuple) else p.predictions
    labels = p.label_ids

    # Apply sigmoid since it's a multi-label setup
    sigmoid = torch.nn.Sigmoid()
    probs = sigmoid(torch.Tensor(predictions))

    # Apply a 0.5 threshold to convert probabilities to binary predictions
    y_pred = np.zeros(probs.shape)
    y_pred[np.where(probs >= 0.5)] = 1

    # Calculate Macro F1, Micro F1, and Exact Match Accuracy
    f1_micro = f1_score(y_true=labels, y_pred=y_pred, average='micro')
    f1_macro = f1_score(y_true=labels, y_pred=y_pred, average='macro', zero_division=0)
    precision = precision_score(y_true=labels, y_pred=y_pred, average='macro')
    recall = recall_score(y_true=labels, y_pred=y_pred, average='macro')
    accuracy = accuracy_score(y_true=labels, y_pred=y_pred)

    return {
        'f1_micro': f1_micro,
        'f1_macro': f1_macro,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall
    }

In [ ]:
# ---- 6. Training Arguments & Trainer Setup

# Using standard hyperparameters for fine-tuning sequence classification models
training_args = TrainingArguments(
    output_dir="./modern_gbert_finetuned",
    eval_strategy="epoch",            # Evaluate at the end of each epoch
    save_strategy="epoch",            # Save a checkpoint at the end of each epoch
    learning_rate=1e-5,               # Standard starting learning rate for BERT
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,               # 5 epochs is generally solid for a small dataset
    weight_decay=0.05,
    warmup_ratio=0.1,                 # Linear warmup for 10% of training steps
    load_best_model_at_end=True,      # Automatically load the best model based on the dev set
    metric_for_best_model="f1_macro", # Optimizing for Macro F1 treats all categories equally
    logging_dir='./logs',
    logging_steps=10
)

trainer = Trainer(
    model=model_regular,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["dev"],
    compute_metrics=compute_metrics
)

In [ ]:
# ---- 7. Train and Evaluate

# Execute the training loop
trainer.train()

# Final evaluation against the unseen test set
print("\n--- Final Test Set Evaluation ---")
test_results = trainer.evaluate(tokenized_datasets["test"])
print(test_results)